# 11.2 Setting up and solving models and data

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

To apply a solver to an instance of a `model`, the examples in this book use `model`,
`data`, and `solve` commands:

```ampl
ampl: model diet.mod; data diet.dat; solve;
MINOS 5.5: optimal solution found.

6 iterations, objective 88.2
```

The `model` command names a file that contains `model` declarations (Chapters 5 through8),
and the `data` command names a file that contains `data` values for `model` components
([Chapter 9](../09/09.md)). The `solve` command causes a description of the optimization problem to
be sent to a solver, and the results to be retrieved for examination. This section takes a
closer look at the main AMPL features for setting up and solving models. Features for
subsequently changing and re-solving models are covered in [Section 11.4](./11_4_modifying_models.ipynb).

**Entering models and `data`**

AMPL maintains a "current" `model`, which is the one that will be sent to the solver if
you type `solve`. At the beginning of an interactive session, the current `model` is empty.
A `model` command reads declarations from a file and adds them to the current `model`; a
`data` command reads `data` statements from a file to supply values for components
already in the current `model`. Thus you may use several `model` or `data` commands to
build up the description of an optimization problem, reading different parts of the `model`
and `data` from different files.

You can also type parts of a `model` and its `data` directly at an AMPL prompt. Model
declarations such as `param`, `var` and `subject to` act as commands that add components
to the current `model`. The `data` statements of [Chapter 9](../09/09.md) also act as commands,
which supply `data` values for already defined components such as sets and parameters.
Because `model` and `data` statements look much alike, however, you need to tell AMPL
which you will be typing. AMPL always starts out in "model mode"; the statement
`data` (without a filename) switches the interpreter to "data mode", and the statement
`model` (without a filename) switches it back. Any command (like option, `solve` or
`subject to`) that does not begin like a `data` statement also has the effect of switching
`data` mode back to `model` mode.

If a `model` declares more than one `objective` function, AMPL by default passes all of
them to the solver. Most solvers deal only with one `objective` function and usually select
the first by default. The `objective` command lets you select a single `objective` function
to pass to the solver; it consists of the keyword `objective` followed by a name
from a minimize or maximize declaration:

```ampl
objective Total_Number;
```

If a `model` has an indexed collection of objectives, you must supply a subscript to indicate
which one is to be chosen:

```ampl
objective Total_Cost["A&P"];
```

The uses of multiple objectives are illustrated by two examples in [Section 8.3](../08/8_3_objectives.ipynb).

**Solving a `model`**

The `solve` command sets in motion a series of activities. First, it causes AMPL to
generate a specific optimization problem from the `model` and `data` that you have supplied.
If you have neglected to provide some needed `data`, an error message is printed; you will
also get error messages if your `data` values violate any restrictions imposed by qualification
phrases in `var` or `param` declarations or by `check` statements. AMPL waits to verify
`data` restrictions until you type `solve`, because a restriction may depend in a complicated
way on many different `data` values. Arithmetic errors like dividing by zero are also
caught at this stage.

After the optimization problem is generated, AMPL enters a "presolve" phase that
tries to make the problem easier for the solver. Sometimes presolve so greatly reduces
the size of a problem that it become substantially easier to `solve`. Normally the work of
presolve goes on behind the scenes, however, and you need not be concerned about it. In
rare cases, presolve can substantially affect the optimal values of the variables — when
there is more than one optimal solution — or can interfere with other preprocessing routines
that are built into your solver software. Also presolve sometimes detects that no
feasible solution is possible, and so does not bother sending your program to the solver.
For example, if you drastically reduce the availability of one resource in steel4.mod,
then AMPL produces an error message:

```ampl
ampl: model steel4.mod;
ampl: data steel4.dat;
ampl: let avail['reheat'] := 10;
ampl: solve;
presolve: constraint Time['reheat'] cannot hold:
       body <= 10 cannot be >= 11.25; difference = -1.25
```

For these cases you should consult the detailed description of presolve in Section 14.1.

The generated optimization problem, as possibly modified by presolve, is finally sent
by AMPL to the solver of your choice. Every version of AMPL is distributed with some
default solver that will be used automatically if you give no other instructions; type
option solver to see its name:

```ampl
ampl: option solver;
option solver minos;
```

If you have more than one solver, you can switch among them by changing the solver
option:

```ampl
ampl: model steelT.mod; data steelT.dat;
ampl: solve;
MINOS 5.5: optimal solution found.

15 iterations, objective 515033
```

```ampl
ampl: reset;
ampl: model steelT.mod;
ampl: data steelT.dat;
ampl: option solver cplex;
ampl: solve;

CPLEX 8.0.0: optimal solution; objective 515033

16 dual simplex iterations (0 in phase I)

ampl: reset;
ampl: model steelT.mod;
ampl: data steelT.dat;
ampl: option solver snopt;
ampl: solve;

SNOPT 6.1-1: Optimal solution found.

15 iterations, objective 515033
```

In this example we `reset` the problem between solves, so that the solvers are invoked with
the same initial conditions and their performance can be compared. Without `reset`, information
about the solution found by one solver would be passed along to the next one,
possibly giving the latter a substantial advantage. Passing information from one `solve`
to the next is most useful when a series of similar LPs are to be sent to the same solver;
we discuss this case in more detail in Section 14.2.

Almost any solver can handle a linear program, although those specifically designed
for linear programming generally give the best performance. Other kinds of optimization
problems, such as nonlinear (Chapter 18) and `integer` (Chapter 20), can be handled only
by solvers designed for them. A message such as "ignoring integrality" or "can't handle
nonlinearities" is an indication that you have not chosen a solver appropriate for your
`model`.

If your optimization problems are not too difficult, you should be able to use AMPL
without referring to instructions for a specific solver: set the solver option
appropriately, type `solve`, and wait for the results.

If your solver takes a very long time to return with a solution, or returns to AMPL
without any "optimal solution" message, then it's time to read further. Each solver is a
sophisticated collection of algorithms and algorithmic strategies, from which many combinations
of choices can be made. For most problems the solver makes good choices
automatically, but you can also pass along your own choices through AMPL options. The
details may vary with each solver, so for more information you must look to the solverspecific
instructions that accompany your AMPL software.

If your problem takes a long time to optimize, you will want some evidence of the
solver's progress to appear on your screen. Directives for this purpose are also described
in the solver-specific instructions.